# 01 - Dataset setup and small fracture subset

Use this notebook first. It prepares a small YOLO detection dataset with exactly one class:

```yaml
names:
  0: fracture
```

Supported sources:
- An existing YOLO-formatted fracture dataset in local, Colab Drive, or Kaggle input storage.
- Optional Roboflow export. Keep API keys in notebook secrets or environment variables; do not paste keys into the repo.

This notebook does not train a model.


In [ ]:
# Runtime mode: "local", "colab", or "kaggle".
RUN_ENV = "local"
PROJECT_NAME = "yolov8-fracture-detection"

# Keep the subset small for quick GPU experiments.
MAX_IMAGES = 240
SEED = 42
VAL_FRACTION = 0.20
TEST_FRACTION = 0.10


In [ ]:
from pathlib import Path
import os
if RUN_ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
elif RUN_ENV == "kaggle":
    PROJECT_ROOT = Path("/kaggle/working") / PROJECT_NAME
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

DATA_ROOT = PROJECT_ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
SUBSET_ROOT = DATA_ROOT / "fracture_subset"
RUNS_ROOT = PROJECT_ROOT / "runs"
WEIGHTS_ROOT = PROJECT_ROOT / "weights"

for folder in (DATA_ROOT, RAW_ROOT, SUBSET_ROOT, RUNS_ROOT, WEIGHTS_ROOT):
    folder.mkdir(parents=True, exist_ok=True)


print(f"Project root: {PROJECT_ROOT}")
print(f"Raw datasets: {RAW_ROOT}")
print(f"Prepared subset: {SUBSET_ROOT}")


## Option A - Use an existing YOLO fracture dataset

Point `SOURCE_DATASET` at a YOLO object detection export. The helper accepts both common layouts:

```text
source_dataset/
  images/train/*.jpg
  images/valid/*.jpg  # or images/val/*.jpg
  labels/train/*.txt
  labels/valid/*.txt  # or labels/val/*.txt
```

```text
source_dataset/
  train/images/*.jpg
  train/labels/*.txt
  valid/images/*.jpg
  valid/labels/*.txt
```

Each label row must be `class x_center y_center width height`, with normalized coordinates. The helper writes a new `fracture.yaml` and remaps copied labels to class `0` so the final subset is single-class.


In [ ]:
# Examples:
# Colab Drive: Path("/content/drive/MyDrive/fracture-data/export")
# Kaggle:      Path("/kaggle/input/my-fracture-yolo-dataset")
# Local:       RAW_ROOT / "my-fracture-yolo-export"
SOURCE_DATASET = RAW_ROOT / "my-fracture-yolo-export"


## Option B - Optional Roboflow download

Use this only if you have a Roboflow project/version exported as YOLO. Store the API key outside the repo:

- Colab: put it in Colab secrets or set it in the session.
- Kaggle: use Kaggle notebook secrets.
- Local: use an untracked `.env` or shell environment variable.

The export should be a fracture-only object detection version. Do not commit Roboflow download URLs or API keys.


In [ ]:
USE_ROBOFLOW = False
ROBOFLOW_WORKSPACE = "your-workspace"
ROBOFLOW_PROJECT = "your-project"
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = "yolov8"  # Use the YOLOv8 export when available; YOLOv5 PyTorch is also YOLO-compatible.

if USE_ROBOFLOW:
    api_key = os.environ.get("ROBOFLOW_API_KEY")
    if not api_key:
        raise RuntimeError("Set ROBOFLOW_API_KEY in your notebook secrets or environment first.")
    from roboflow import Roboflow

    rf = Roboflow(api_key=api_key)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    downloaded = version.download(model_format=ROBOFLOW_FORMAT, location=str(RAW_ROOT / "roboflow-fracture"))
    SOURCE_DATASET = Path(downloaded.location)

SOURCE_DATASET


## Build the small single-class subset

The source should already be fracture-focused. If your source has multiple medical labels, filter it first in the source system or create a fracture-only Roboflow version before running this cell.


In [ ]:
import random
import shutil
from dataclasses import dataclass

IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}

@dataclass(frozen=True)
class DatasetCounts:
    train_images: int
    val_images: int
    test_images: int
    train_labels: int
    val_labels: int
    test_labels: int


def write_fracture_yaml(dataset_root, yaml_path=None):
    root = Path(dataset_root).expanduser().resolve()
    target = Path(yaml_path).expanduser().resolve() if yaml_path else root / "fracture.yaml"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        "\n".join([
            f"path: {root.as_posix()}",
            "train: images/train",
            "val: images/val",
            "test: images/test",
            "names:",
            "  0: fracture",
            "",
        ]),
        encoding="utf-8",
    )
    return target


def summarize_yolo_dataset(dataset_root):
    root = Path(dataset_root).expanduser()
    counts = []
    for kind in ("images", "labels"):
        for split in ("train", "val", "test"):
            folder = root / kind / split
            if kind == "images":
                counts.append(sum(1 for item in folder.glob("*") if item.suffix.lower() in IMAGE_EXTENSIONS))
            else:
                counts.append(sum(1 for item in folder.glob("*.txt")))
    return DatasetCounts(*counts)


def validate_single_class_labels(dataset_root):
    root = Path(dataset_root).expanduser()
    bad_lines = []
    for label_path in sorted((root / "labels").glob("**/*.txt")):
        for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
            stripped = line.strip()
            if not stripped:
                continue
            parts = stripped.split()
            if len(parts) != 5 or parts[0] != "0":
                bad_lines.append(f"{label_path}:{line_number}: {stripped}")
    if bad_lines:
        preview = "\n".join(bad_lines[:10])
        raise ValueError(f"Labels must be YOLO detect rows with class id 0 only:\n{preview}")


def candidate_split_dirs(source, split):
    source = Path(source)
    return (
        (source / "images" / split, source / "labels" / split),
        (source / split / "images", source / split / "labels"),
    )


def collect_labeled_images(source_root):
    source = Path(source_root).expanduser()
    examples = []
    for split in ("train", "valid", "val", "test"):
        for image_dir, label_dir in candidate_split_dirs(source, split):
            if not image_dir.exists() or not label_dir.exists():
                continue
            for image_path in image_dir.iterdir():
                if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                label_path = label_dir / f"{image_path.stem}.txt"
                if label_path.exists() and label_path.read_text(encoding="utf-8").strip():
                    examples.append((image_path, label_path))
    return examples


def split_examples(examples, val_fraction, test_fraction):
    total = len(examples)
    test_count = max(1, round(total * test_fraction))
    val_count = max(1, round(total * val_fraction))
    if test_count + val_count >= total:
        test_count = 1
        val_count = 1
    test_items = examples[:test_count]
    val_items = examples[test_count:test_count + val_count]
    train_items = examples[test_count + val_count:]
    return train_items, val_items, test_items


def copy_label_as_fracture_only(source, target, remap_all_labels_to_fracture=True):
    rows = []
    for line in Path(source).read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        if len(parts) != 5:
            raise ValueError(f"Expected YOLO detect label with 5 columns in {source}: {line}")
        if parts[0] != "0" and not remap_all_labels_to_fracture:
            raise ValueError(f"Found class id {parts[0]} in {source}; expected class id 0 only")
        rows.append(" ".join(["0", *parts[1:]]))
    Path(target).write_text("\n".join(rows) + ("\n" if rows else ""), encoding="utf-8")


def prepare_small_fracture_subset(
    source_root,
    output_root,
    max_images=240,
    seed=42,
    val_fraction=0.2,
    test_fraction=0.1,
    remap_all_labels_to_fracture=True,
):
    source = Path(source_root).expanduser()
    output = Path(output_root).expanduser()
    if not source.exists():
        raise FileNotFoundError(f"Source dataset does not exist: {source}")
    if not 0 <= val_fraction < 1 or not 0 <= test_fraction < 1 or val_fraction + test_fraction >= 1:
        raise ValueError("val_fraction and test_fraction must be non-negative and sum to less than 1")
    if max_images < 3:
        raise ValueError("max_images must be at least 3 so train/val/test can be populated")

    examples = collect_labeled_images(source)
    if len(examples) < 3:
        raise ValueError("Need at least three labeled fracture images to create train/val/test splits")

    random.Random(seed).shuffle(examples)
    selected = examples[:min(max_images, len(examples))]
    train_items, val_items, test_items = split_examples(selected, val_fraction, test_fraction)

    if output.exists():
        shutil.rmtree(output)
    for split in ("train", "val", "test"):
        (output / "images" / split).mkdir(parents=True, exist_ok=True)
        (output / "labels" / split).mkdir(parents=True, exist_ok=True)

    for split, items in (("train", train_items), ("val", val_items), ("test", test_items)):
        for image_path, label_path in items:
            shutil.copy2(image_path, output / "images" / split / image_path.name)
            copy_label_as_fracture_only(
                label_path,
                output / "labels" / split / f"{image_path.stem}.txt",
                remap_all_labels_to_fracture,
            )

    return write_fracture_yaml(output)


In [ ]:
yaml_path = prepare_small_fracture_subset(
    source_root=SOURCE_DATASET,
    output_root=SUBSET_ROOT,
    max_images=MAX_IMAGES,
    seed=SEED,
    val_fraction=VAL_FRACTION,
    test_fraction=TEST_FRACTION,
    remap_all_labels_to_fracture=True,
)
validate_single_class_labels(SUBSET_ROOT)
counts = summarize_yolo_dataset(SUBSET_ROOT)
print(counts)
print(f"Dataset YAML: {yaml_path}")


In [ ]:
print(Path(yaml_path).read_text())


## Next step

Open `02_train_yolov8_fracture.ipynb`, use the same `RUN_ENV`, and point `DATA_YAML` to the printed YAML path.
